In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import os


CONTINUOUS_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_ms'
]
DISCRETE_FEATURES = ['key', 'mode', 'time_signature', 'explicit']
ID_COLS           = ['track_id', 'track_name', 'artists', 'album_name']

# Log transform features
LOG_FEATURES = ['speechiness', 'liveness', 'instrumentalness', 'duration_ms']

df = pd.read_csv('../data/raw/spotify_dataset.csv', index_col=0)
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
# We don't need the ID_COLS since they won't provide any predictive advantage.
df_cleaned = df.drop(columns=ID_COLS)
print(f'Cleaned shape after drop: {df_cleaned.shape}')

Cleaned shape after drop: (114000, 16)


<h2>Duplicate Removal</h2>

**Motivation**
Strum (2014) Models could output falsely high accuracy by memorizing repeated tracks rather than learning the actual characteristics of the genres. EDA found 26.77% of audio features share attributes.

In [4]:
# Row Duplicates
rows_before = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
print(f'Duplicated rows removed: {rows_before - len(df_cleaned)}')
print(f'Shape after dropping: {df_cleaned.shape}')

Duplicated rows removed: 7093
Shape after dropping: (106907, 16)


In [5]:
'''
Removal of the same song under multiple genres.
Keeping the first occurrence
'''
features_before = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates(subset=CONTINUOUS_FEATURES, keep='first')
print(f'Duplicated features removed: {features_before - len(df_cleaned)}')
print(f'Duplicated rows and feature removed: {rows_before - len(df_cleaned)}')
print(f'Remaining : {len(df_cleaned)}')
df_cleaned.shape


Duplicated features removed: 23428
Duplicated rows and feature removed: 30521
Remaining : 83479


(83479, 16)

In [6]:
null_counts = df_cleaned.isnull().sum()
print(f'Null counts:')
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No Nulls.')

Null counts:
No Nulls.


In [7]:
for feat in CONTINUOUS_FEATURES:
    num_nulls = df_cleaned[feat].isnull().sum()
    if num_nulls > 0:
        median_ = df_cleaned[feat].median()
        df_cleaned[feat].fillna(median_, inplace=True)
        print(f'Imputed {feat} with median: {median_:.4f} ({num_nulls} values)')

# Drop rows where there's no target variable
next_before = len(df_cleaned)
df_cleaned = df_cleaned.dropna(subset=['track_genre'])
print(f'Rows dropped with track genre label missing: {next_before - len(df_cleaned)}')


Rows dropped with track genre label missing: 0


<h2>Log Transform</h2>

**Motivation**

This is only applied to these features:

<b>speechiness, liveness, instrumentalness, duration_ms</b>

This will reduces large value influence for distance-based models
but wont be needed for tree based models if used (SVM or k-means)

Consistent with feature preparation best practices in audio ML (Li et al., 2003; Kim et al., 2010)
We are using log1p(x) to handle zero values by shifting input by 1 and avoiding NaN or -inf being input
into model calculations.

In [10]:
df_clean_with_log = df_cleaned.copy()

for feat in LOG_FEATURES:
    df_clean_with_log[feat] = np.log1p(df_clean_with_log[feat])
    print(f'Log transform applied to feature: {feat}')
    print(f'  before transform: mean={df_cleaned[feat].mean():.4f}, skew={df_cleaned[feat].skew():.4f}')
    print(f'  after transform: mean={df_clean_with_log[feat].mean():.4f}, skew={df_clean_with_log[feat].skew():.4f}')
    print()

Log transform applied to feature: speechiness
  before transform: mean=0.0885, skew=4.5018
  after transform: mean=0.0804, skew=3.6485

Log transform applied to feature: liveness
  before transform: mean=0.2192, skew=2.0376
  after transform: mean=0.1872, skew=1.6891

Log transform applied to feature: instrumentalness
  before transform: mean=0.1814, skew=1.4921
  after transform: mean=0.1351, skew=1.4164

Log transform applied to feature: duration_ms
  before transform: mean=231252.8901, skew=11.0663
  after transform: mean=12.2703, skew=-0.6695



## Log Transform Results

| Feature | Skew Before | Skew After | Result                                            |
|---|---|---|---------------------------------------------------|
| duration_ms | 11.07 | -0.67 | Excellent -> essentially normalized               |
| speechiness | 4.50 | 3.65 | Moderate -> still skewed, zero-inflated           |
| liveness | 2.04 | 1.69 | Partial -> acceptable for modeling                |
| instrumentalness | 1.49 | 1.42 | Minimal -> bimodal distribution resists transform |

<h2>Feature Scaling</h2>

Three versions created for different model types

StandardScaler -> SVM or k-means

MinMaxScaler -> Neural Networks

Just the raw df_cleaned -> Tree based (Random Forest, AdaBoost etc)

In [11]:
scaler_std = StandardScaler()
scaler_min_max = MinMaxScaler()

df_std_scale = df_clean_with_log.copy()
df_std_scale[CONTINUOUS_FEATURES] = scaler_std.fit_transform(df_clean_with_log[CONTINUOUS_FEATURES])

df_min_max = df_clean_with_log.copy()
df_min_max[CONTINUOUS_FEATURES] = scaler_min_max.fit_transform(df_clean_with_log[CONTINUOUS_FEATURES])

joblib.dump(scaler_std, '../outputs/models/standard_scaler.pkl')
joblib.dump(scaler_min_max, '../outputs/models/min_max_scaler.pkl')

['../outputs/models/min_max_scaler.pkl']

<h2>Popularity Binarization</h2>

**Motivation**

Middlebrook & Sheik (2019):

Binary classification of popularity. We will use a median as threshold
since we aren't using the Billboard Top 100 as Middlebrook & Sheik did in thier research

In [28]:
popular_median = df_cleaned['popularity'].median()
df_cleaned['popular'] = (df_cleaned['popularity'] >= popular_median).astype(int)
df_std_scale['popular'] = df_cleaned['popular']
df_min_max['popular'] = df_cleaned['popular']
df_clean_with_log['popular'] = df_cleaned['popular']

print(f'Popularity median threshold: {popular_median:.4f}')
print(f'Labeled Popular: {df_cleaned['popular'].sum()} '
      f'  Pct of Dataset({df_cleaned['popular'].mean() * 100:.1f}%)')
print(f'Not Labeled Popular: {(df_cleaned['popular'] == 0).sum()} '
      f'  Pct of Dataset({(1-df_cleaned['popular'].mean()) * 100:.1f}%)')


Popularity median threshold: 35.0000
Labeled Popular: 42181   Pct of Dataset(50.5%)
Not Labeled Popular: 41298   Pct of Dataset(49.5%)


## Popularity Binarization Result

- Median threshold: 35
- Labeled Popular: 42,181 tracks (50.5%)
- Not popular: 41,298 tracks (49.5%)
- Class balance is good

Almost 50/50 split confirms that median is the appropriate
threshold for binary classification in Song Popularity Prediction
in Objective 2. This is consistent with Middlebrook & Sheik (2019)
who also used balanced hit/non-hit datasets for popularity classification.

<h2>Train/Validation/Test Splits</h2>

- Stratified 70/10/20 split on track_genre
- Ensures all 114 genres proportionally represented in every split
- All modeling can use these splits

Note: Split justification in eda notebook

In [36]:
X = df_cleaned.drop(columns=['track_genre','popularity','popular'])
y_genre = df_cleaned['track_genre'] # multiclass classification target
y_popular_regress = df_cleaned['popularity'] # popularity score regression target
y_popular_classify = df_cleaned['popular'] # binary classification target

# 80% Train Validation / 20% Test
X_trainval, X_test, y_genre_trainval, y_genre_test = train_test_split(
    X,
    y_genre,
    test_size=0.20,
    random_state=42,
    stratify=y_genre)

# From 80% taking 12.5% as Validation (10% of the remaining dataset)
X_train, X_val, y_genre_train, y_genre_val = train_test_split(
    X_trainval,
    y_genre_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_genre_trainval
)

# Validating Splits
print(f'Train: {len(X_train):>6} Rows: ({len(X_train)/len(X)*100:.1f}%)')
print(f'Validation: {len(X_val):>6} Rows: ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test: {len(X_test):>6} Rows: ({len(X_test)/len(X)*100:.1f}%)')

Train:  58435 Rows: (70.0%)
Validation:   8348 Rows: (10.0%)
Test:  16696 Rows: (20.0%)


In [37]:
# Popularity targets should be from the same splits. Indexing should be same for both regression and binary targets
# This should mitigate any data shape issues later. For Objective 2

y_popular_reg_train = y_popular_regress.loc[X_train.index]
y_popular_reg_val = y_popular_regress.loc[X_val.index]
y_popular_reg_test = y_popular_regress.loc[X_test.index]

y_popular_class_train = y_popular_classify.loc[X_train.index]
y_popular_class_val = y_popular_classify.loc[X_val.index]
y_popular_class_test = y_popular_classify.loc[X_test.index]

In [38]:
# Raw clean — tree-based models
df_cleaned.to_csv('../data/processed/spotify_clean.csv')

# Log transformed — reference version
df_clean_with_log.to_csv('../data/processed/spotify_log_transformed.csv')

# Scaled versions
df_std_scale.to_csv('../data/processed/spotify_standard_scaled.csv')
df_min_max.to_csv('../data/processed/spotify_minmax_scaled.csv')

# Feature splits (from raw clean)
X_train.to_csv('../data/processed/X_train.csv')
X_val.to_csv('../data/processed/X_val.csv')
X_test.to_csv('../data/processed/X_test.csv')

# Genre labels: Objective 1
y_genre_train.to_csv('../data/processed/y_genre_train.csv')
y_genre_val.to_csv('../data/processed/y_genre_val.csv')
y_genre_test.to_csv('../data/processed/y_genre_test.csv')

# Popularity regression: Objective 2
y_popular_reg_train.to_csv('../data/processed/y_popular_reg_val.csv')
y_popular_reg_val.to_csv('../data/processed/y_popular_reg_val.csv')
y_popular_reg_test.to_csv('../data/processed/y_popular_reg_test.csv')

# Popularity binary: Objective 2
y_popular_class_train.to_csv('../data/processed/y_popular_class_train.csv')
y_popular_class_val.to_csv('../data/processed/y_popular_class_val.csv')
y_popular_class_test.to_csv('../data/processed/y_popular_class_test.csv')
